In [1]:
import cobra
import pycomo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pycomo.configure_logger(level="error")

2026-08-07 15:55:43,916 - PyCoMo - INFO - Logger initialized.


In [2]:
models = ["Rhodoferax", "Geobacter"]
#for modelname in models:
    # model = cobra.io.load_matlab_model(f"models/{modelname}.mat")

    # if modelname == "Geobacter":
    #     to_remove = [rxn.id for rxn in model.reactions if rxn.id.startswith("DM_")]
    #     model.remove_reactions(to_remove)
    #     model.reactions.ATPM.bounds = (0.,1000.)

    # cobra.io.write_sbml_model(model, f"models/original_models/{modelname}.xml")

In [3]:
c_sources = ["EX_mal-L_e", "EX_cit_e", "EX_fum_e"]

dimes = {"Rhodoferax": {
            "mEmC_1": {"uptakes": ["EX_nh4_e", "EX_fe2_e", "EX_fum_e", "EX_o2_e"],
                        "secretion": ["EX_mal-L_e",  "EX_co2_e"]},
            "mEmC_3": {"uptakes": ["EX_nh4_e", "EX_fe2_e", "EX_fum_e", "EX_o2_e"],
                        "secretion": ["EX_mal-L_e",  "EX_co2_e"]},
    
            "mEmC_2": {"uptakes": ["EX_nh4_e", "EX_fe2_e", "EX_cit_e", "EX_o2_e"],
                "secretion": ["EX_mal-L_e",  "EX_co2_e"]},
            "mEmC_4": {"uptakes": ["EX_nh4_e", "EX_fe2_e", "EX_cit_e", "EX_o2_e"],
                "secretion": ["EX_mal-L_e",  "EX_co2_e"]},

            "mEmC_5": {"uptakes": ["EX_nh4_e", "EX_fe2_e", "EX_mal-L_e", "EX_o2_e"],
                "secretion": ["EX_co2_e"]},
            "mEmC_6": {"uptakes": ["EX_nh4_e", "EX_fe2_e", "EX_mal-L_e", "EX_o2_e",
                                  "EX_co2_e"],
                      "secretion": []},
            "mE_1": {"uptakes": ["EX_nh4_e", "EX_ac_e", "EX_fe3_e", "EX_o2_e"],
                    "secretion": ["EX_co2_e"]},
},
        "Geobacter": {
            "mEmC_1": {"uptakes": ["EX_nh4_e", "EX_fe3_e", "EX_mal-L_e"],
                      "secretion": ["EX_fe2_e", "EX_co2_e", "EX_h2s_e"]},
            "mEmC_2": {"uptakes": ["EX_nh4_e", "EX_fe3_e", "EX_mal-L_e"],
                      "secretion": ["EX_fe2_e", "EX_co2_e", "EX_h2s_e"]},
            "mEmC_3": {"uptakes": ["EX_n2_e", "EX_fe3_e", "EX_mal-L_e"],
                      "secretion": ["EX_fe2_e", "EX_co2_e", "EX_h2s_e"]},
            "mEmC_4": {"uptakes": ["EX_n2_e", "EX_fe3_e", "EX_mal-L_e"],
                      "secretion": ["EX_fe2_e", "EX_co2_e", "EX_h2s_e"]},
            "mEmC_5": {"uptakes": ["EX_n2_e", "EX_fe3_e", "EX_cit_e"],
                      "secretion": ["EX_fe2_e", "EX_mal-L_e", "EX_co2_e", "EX_h2s_e"]},
            "mEmC_6": {"uptakes": ["EX_n2_e", "EX_fe3_e", "EX_cit_e"],
                      "secretion": ["EX_fe2_e", "EX_mal-L_e", "EX_co2_e", "EX_h2s_e"]},
            "mE_1": {"uptakes": ["EX_n2_e", "EX_fe3_e", "EX_mal-L_e"],
                  "secretion": ["EX_fe2_e", "EX_ac_e", "EX_co2_e", "EX_h2s_e"]},
    }}

crossfed = {"mEmC_1": ["EX_mal-L_e", "EX_fe2_e"],
          "mEmC_2": ["EX_mal-L_e", "EX_fe2_e"],
          "mEmC_3": ["EX_mal-L_e", "EX_fe2_e"],
          "mEmC_4": ["EX_mal-L_e", "EX_fe2_e"],
          "mEmC_5": ["EX_mal-L_e", "EX_fe2_e"],
          "mEmC_6": ["EX_mal-L_e", "EX_co2_e",  "EX_fe2_e"],
          "mE_1": ["EX_ac_e"]}

main_c_source = {"mEmC_1": {"c_source": "fum", "feeder": "Rhodoferax"},
                 "mEmC_2": {"c_source": "cit", "feeder": "Rhodoferax"},
                 "mEmC_3": {"c_source": "fum", "feeder": "Rhodoferax"},
                 "mEmC_4": {"c_source": "cit", "feeder": "Rhodoferax"},
                 "mEmC_5": {"c_source": "cit", "feeder": "Geobacter"},
                 "mEmC_6": {"c_source": "cit", "feeder": "Geobacter"},
                 "mE_1":   {"c_source": "mal", "feeder": "Geobacter"}}

unique_uptakes = {"Geobacter": {}, "Rhodoferax": {}}
for pattern in crossfed:
    geo = dimes["Geobacter"][pattern]["uptakes"]
    rho = dimes["Rhodoferax"][pattern]["uptakes"]

    unique_uptakes["Geobacter"][pattern] = set(geo).difference(set(rho))
    unique_uptakes["Rhodoferax"][pattern] = set(rho).difference(set(geo))

In [4]:
unique_uptakes

{'Geobacter': {'mEmC_1': {'EX_fe3_e', 'EX_mal-L_e'},
  'mEmC_2': {'EX_fe3_e', 'EX_mal-L_e'},
  'mEmC_3': {'EX_fe3_e', 'EX_mal-L_e', 'EX_n2_e'},
  'mEmC_4': {'EX_fe3_e', 'EX_mal-L_e', 'EX_n2_e'},
  'mEmC_5': {'EX_cit_e', 'EX_fe3_e', 'EX_n2_e'},
  'mEmC_6': {'EX_cit_e', 'EX_fe3_e', 'EX_n2_e'},
  'mE_1': {'EX_mal-L_e', 'EX_n2_e'}},
 'Rhodoferax': {'mEmC_1': {'EX_fe2_e', 'EX_fum_e', 'EX_o2_e'},
  'mEmC_2': {'EX_cit_e', 'EX_fe2_e', 'EX_o2_e'},
  'mEmC_3': {'EX_fe2_e', 'EX_fum_e', 'EX_nh4_e', 'EX_o2_e'},
  'mEmC_4': {'EX_cit_e', 'EX_fe2_e', 'EX_nh4_e', 'EX_o2_e'},
  'mEmC_5': {'EX_fe2_e', 'EX_mal-L_e', 'EX_nh4_e', 'EX_o2_e'},
  'mEmC_6': {'EX_co2_e', 'EX_fe2_e', 'EX_mal-L_e', 'EX_nh4_e', 'EX_o2_e'},
  'mE_1': {'EX_ac_e', 'EX_nh4_e', 'EX_o2_e'}}}

In [5]:
model_dir = "models/original_models"
named_models = pycomo.load_named_models_from_dir(model_dir)

In [6]:
single_org_models = []
for modelname, model in named_models.items():
    print(modelname, model.slim_optimize())

    single_org_model = pycomo.SingleOrganismModel(model, modelname)
    single_org_models.append(single_org_model)

Rhodoferax 41.91559360486998
Geobacter 35.65356530827624


In [7]:
def get_medium_for_pattern(pattern, dimes, 
                           c_sources, crossfed, models):
    medium = {
        'EX_so4_medium': 1000.,
        'EX_pi_medium': 1000.,
        'EX_mg2_medium': 1000.,
        'EX_k_medium': 1000.,
        'EX_ca2_medium': 1000.,
        'EX_h_medium': 1000.,
        'EX_h2_medium': 1000.,
        'EX_h2o_medium': 1000.,
        'EX_nh4_medium': 1000.}

    for modelname in models:
        for component in dimes[modelname][pattern]["uptakes"]:
            max_uptake = 1000.
            if component in c_sources:
                max_uptake = 10.
            if component in crossfed[pattern]:
                continue
    
            medium_name = f"{component.replace('_e', '').replace('-L', '_L')}_medium"
            medium[medium_name] = max_uptake
    return medium

In [8]:
def define_unique_uptakes(pattern, unique_uptakes, com_model):
    off = []
    
    for org in unique_uptakes:
        unique = unique_uptakes[org][pattern]
        for rxn in unique:
            met = rxn.split("_")[1].replace("-", "_")
    
            if org == "Geobacter":
                off_for = "Rhodoferax"
            else:
                off_for = "Geobacter"
    
            transfer_id = f"{off_for}_TF_{met}_{off_for}_e"
            
            try:
                off.append(com_model.model.reactions.get_by_id(transfer_id).id)
                print(f"{transfer_id} off")
            except KeyError:
                print(f"{transfer_id} not found")
    
    for rxn_id in off:
        com_model.model.reactions.get_by_id(rxn_id).lower_bound = 0

In [9]:
def turn_off_unwanted_transfer_rxns(com_model, crossfed, pattern, medium):
    """turn off transfer reactions for mets not in medium and not crossfed
    to enforce a specific exchange pattern"""
    for t in com_model.transfer_reactions:
        met = t.id.split("_")[2]
        found = any(met in ex for ex in crossfed[pattern]) 
        found2 = any(met in ex for ex in medium) 
    
        if (not found) and (not found2):
            t.lower_bound = 0
            #print(f"{t.id} turned off")
        else:
            pass
            #print(f"{met} can be taken up") 

In [10]:
def get_compositions(com_model, mus, organism="Geobacter"):
    mins = []
    maxs = []
    feasible_mus = []

    for mu in mus:
        try:
            comps = com_model.feasible_composition_range(mu)
        except Exception as e:
            print(f"Growth rate {mu} infeasible")
            break
            
        feasible_mus.append(mu)
        geo = comps.loc[comps.reaction_id == f"{organism}_fraction_reaction"]
        mins.append(geo.min_flux.item())
        maxs.append(geo.max_flux.item())

    return feasible_mus, mins, maxs

def plot_compositions(mus, mins, maxs, organism="Geobacter", title=""):
    color = "#00a797" # "#d43689"

    plt.fill_betweenx(mus, mins, maxs, alpha = 0.5, color = color)
    plt.xlabel(f"Fraction of {organism}", size=15)
    plt.ylabel("Growth rate", size=15)
    plt.title(title)
    plt.savefig(f"figures/{title}.png", dpi=300, bbox_inches="tight")
    plt.show()


In [11]:
community_name = "uranium_community"
com_model_orig = pycomo.CommunityModel(single_org_models, community_name)
com_model_orig.model.solver = "cplex"

In [12]:
com_model_orig.model.tolerance = 1e-8

In [13]:
com_model_orig.max_growth_rate(return_abundances=True, minimal_abundance=0.1)

2026-08-07 15:55:50,861 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:55:52,147 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:55:54,094 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:55:55,366 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:55:57,044 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:55:58,505 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:56:00,288 - PyCoMo - INFO - Logger initialized.
2026-08-07 15:56:01,570 - PyCoMo - INFO - Logger initialized.


,reaction_id,min_flux,max_flux
0,Rhodoferax_fraction_reaction,0.000000,1.000000
1,Geobacter_fraction_reaction,0.000000,1.000000
2,community_biomass,41.915593,41.915593


In [ ]:
uptake_main = 0.1
results = [] 

for pattern in crossfed:
    com_model = pycomo.CommunityModel(single_org_models, community_name)
    com_model.model.solver = "cplex"
    
    feeder = main_c_source[pattern]["feeder"]
    c_source = main_c_source[pattern]["c_source"]
    
    medium = get_medium_for_pattern(pattern, dimes, 
                                    c_sources, crossfed, models)
    
    com_model.medium = medium
    com_model.apply_medium()
    
    define_unique_uptakes(pattern, unique_uptakes, com_model)
    turn_off_unwanted_transfer_rxns(com_model, crossfed, pattern, medium)
    
    # set uptake rate for main carbon source
    bounds_rxn = com_model.model.reactions.get_by_id(f"{feeder}_fraction_reaction")
    for met, stoich in bounds_rxn.metabolites.items():
        if c_source in met.id:
            bounds_rxn.add_metabolites({met: uptake_main}, combine=False)
    
    fba = com_model.max_growth_rate(return_abundances=True, minimal_abundance=0)
    max_mu = fba.loc[fba.reaction_id == "community_biomass"].max_flux.item()
    
    tested_mus = np.linspace(0.4, max_mu, 10)
    mus, mins, maxs = get_compositions(com_model, tested_mus)
    plot_compositions(mus, mins, maxs, title=pattern)

    # collect rows for this pattern
    for mu, lo, hi in zip(mus, mins, maxs):
        results.append({
            "pattern": pattern,
            "mu": mu,
            "min_fraction": lo,
            "max_fraction": hi,
        })

    com_model.apply_fixed_growth_rate(max_mu*0.9)

    com_model.model.objective = "Geobacter_fraction_reaction"
    fluxes = com_model.model.optimize()
    f = fluxes.fluxes[fluxes.fluxes.index.str.startswith("Geobacter_TF")]
    print("Geobacter fluxes")
    print(f[f < 0])#/geo_f
    print(f[f > 0])
    f = fluxes.fluxes[fluxes.fluxes.index.str.startswith("Rhodoferax_TF")]
    print("Rhodoferax fluxes")
    print(f[f < 0])#
    print(f[f > 0])
    
results_df = pd.DataFrame(results)
results_df.to_csv("results/pycomo_composition_ranges.csv", index=False)

Rhodoferax_TF_mal_L_Rhodoferax_e off
Rhodoferax_TF_fe3_Rhodoferax_e off
Geobacter_TF_fum_Geobacter_e off
Geobacter_TF_fe2_Geobacter_e off
Geobacter_TF_o2_Geobacter_e not found


2026-08-07 16:06:05,102 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:06,495 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:08,687 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:10,036 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:11,957 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:13,260 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:15,231 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:16,557 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:18,492 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:20,124 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:22,094 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:23,383 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:25,306 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:26,609 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:28,559 - PyCoMo - INFO - Logger initialized.
2026-08-07 16:06:29,864 - PyCoMo - INFO - Logger initialized.
2026-08-

In [19]:
for index, f in fluxes.fluxes[abs(fluxes.fluxes) > 0].items():
    if "Geobacter" in index:
        print(index, f)

Geobacter_ATPS4 2.6068178250000004e-06
Geobacter_CYOR1m 1.0427271300000001e-05
Geobacter_FBP_Geobacter_c 2.6068178250000004e-06
Geobacter_FERCYT 2.0854542600000003e-05
Geobacter_H2td -1.0427271300000001e-05
Geobacter_HDH2 1.0427271300000001e-05
Geobacter_PFK_Geobacter_c 2.6068178250000004e-06
Geobacter_TF_h_Geobacter_e 2.0854542600000003e-05
Geobacter_TF_fe2_Geobacter_e 2.0854542600000003e-05
Geobacter_TF_fe3_Geobacter_e -2.0854542600000003e-05
Geobacter_TF_h2_Geobacter_e -1.0427271300000001e-05
SK_Geobacter_ATPS4_ub -2.6068178250000004e-08
SK_Geobacter_CYOR1m_ub -1.0427271300000001e-07
SK_Geobacter_FBP_Geobacter_c_ub -2.6068178250000004e-08
SK_Geobacter_FERCYT_ub -2.0854542600000003e-07
SK_Geobacter_H2td_lb -1.0427271300000001e-07
SK_Geobacter_H2td_ub 1.0427271300000001e-07
SK_Geobacter_HDH2_ub -1.0427271300000001e-07
SK_Geobacter_PFK_Geobacter_c_ub -2.6068178250000004e-08
SK_Geobacter_TF_h_Geobacter_e_lb 2.0854542600000003e-07
SK_Geobacter_TF_h_Geobacter_e_ub -2.0854542600000003e-07


In [21]:
for index, f in fluxes.fluxes[abs(fluxes.fluxes) > 0].items():
    if "SK_" in index:
        continue
    if "Rhodoferax" in index:
        print(index, f)

Rhodoferax_ALATA_L_Rhodoferax_c -0.3693364029216
Rhodoferax_ASNS1_Rhodoferax_c 0.14979940623360002
Rhodoferax_ASPTRS_Rhodoferax_c 0.14979940623360002
Rhodoferax_ALARi_Rhodoferax_c 0.03206692609200001
Rhodoferax_ALATRS_Rhodoferax_c 0.3214445592096
Rhodoferax_A5PISO_Rhodoferax_c 0.027846948060000005
Rhodoferax_ACALDi_Rhodoferax_c 1.2374521279451796
Rhodoferax_GALU_Rhodoferax_c 0.012978885924000004
Rhodoferax_GLYCTO2_Rhodoferax_c 0.0306684450000001
Rhodoferax_PGMT_Rhodoferax_c -0.11099523614400003
Rhodoferax_ACGK_Rhodoferax_c 0.21013527818880004
Rhodoferax_ACOTA_Rhodoferax_c -0.21013527818880004
Rhodoferax_ADCL_Rhodoferax_c 0.0306684450000001
Rhodoferax_ADCS_Rhodoferax_c 0.0306684450000001
Rhodoferax_AGPR_Rhodoferax_c -0.21013527818880004
Rhodoferax_ANPRT_Rhodoferax_c 0.034329030595200004
Rhodoferax_ANS1_Rhodoferax_c 0.034329030595200004
Rhodoferax_ARGTRS_Rhodoferax_c 0.18412843682880004
Rhodoferax_ASPK_Rhodoferax_c 1.9057371722999799
Rhodoferax_ASPTA1_Rhodoferax_c -3.1035442013936674
Rho

In [ ]:
fba.loc[fba.reaction_id == "community_biomass"].max_flux.item()

In [ ]:
#com_model.summary()

In [ ]:
# rho_f = fluxes.fluxes.loc["Rhodoferax_to_community_biomass"]
# geo_f = fluxes.fluxes.loc["Geobacter_to_community_biomass"]

In [ ]:
# f = fluxes.fluxes[fluxes.fluxes.index.str.startswith("Geobacter_TF")]
# f[f < -0.1]#/geo_f

In [ ]:
# f[f > 0.1]#/geo_f

In [ ]:
# r = fluxes.fluxes[fluxes.fluxes.index.str.startswith("Rhodoferax_TF")]
# r[r > 0.1]#/rho_f

In [ ]:
# r[r < -0.1]#/rho_f